# Multiple Linear Regression

**What this notebook covers:**
- Extending Simple Linear Regression to multiple features
- Implementing MLR from scratch using the Normal Equation: β = (XᵀX)⁻¹Xᵀy
- Detecting multicollinearity using correlation heatmaps and VIF
- Comparing Adjusted R² vs plain R² for model evaluation
- Feature count experiments and their effect on model performance

**Prerequisites:**
- Simple Linear Regression (OLS, β interpretation)
- Introduction to Regression Algorithms
- Basic linear algebra: matrix multiplication, matrix inverse
- Python, NumPy, Pandas, Scikit-learn basics

---

**Dataset:** House Prices — Advanced Regression Techniques  
🔗 [Kaggle Dataset Link](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)  
*Credits: Kaggle / Dean De Cock — Ames Housing dataset. We now use multiple features simultaneously, upgrading from Topic 3's single-feature SLR to full MLR.*

In [ ]:
# --- All Imports ---
import numpy as np                        # Matrix operations and Normal Equation
import pandas as pd                       # Data loading and manipulation
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Correlation heatmap and visualizations
from sklearn.linear_model import LinearRegression          # Sklearn MLR
from sklearn.model_selection import train_test_split, cross_val_score  # Splitting and CV
from sklearn.preprocessing import StandardScaler           # Feature scaling
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Metrics
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # Reproducibility for all random operations

## Part 1: Theory Recap

Five key things to remember about Multiple Linear Regression:

- **MLR extends SLR to p features**: ŷ = β₀ + β₁x₁ + ... + βₚxₚ, or in matrix form ŷ = Xβ. Each βⱼ is the partial effect of xⱼ on y, holding all other features constant.
- **Normal Equation solves for all coefficients at once**: β = (XᵀX)⁻¹Xᵀy — one matrix operation gives the optimal β vector that minimizes the sum of squared residuals.
- **Multicollinearity is the biggest danger in MLR**: When features are highly correlated, (XᵀX) becomes near-singular, making (XᵀX)⁻¹ blow up and coefficients unstable. Detect with VIF or correlation heatmap.
- **Adjusted R² is fairer than R² for MLR**: Plain R² always increases when you add features. Adjusted R² penalizes extra features and only increases if they genuinely help.
- **More features ≠ better model**: Adding irrelevant features increases variance without reducing bias. Feature selection (Lasso, correlation analysis, domain knowledge) is critical.

## Loading the Dataset

We use the **Ames Housing dataset** — same as Topic 3, but now with **multiple features**.
- **Input features (X):** 8 key numerical features describing house characteristics
- **Target variable (y):** `SalePrice` — house sale price in USD

Using the same dataset lets us directly compare: SLR (1 feature, Topic 3) vs MLR (8 features, Topic 4).

In [ ]:
# Load the Ames Housing dataset
df = pd.read_csv("train.csv")

# Select 8 key numerical features — chosen for relevance to house price
features = [
    'Gr Liv Area',      # Above ground living area (sq ft)
    'Overall Qual',     # Overall material and finish quality (1-10)
    'Year Built',       # Original construction year
    'Total Bsmt SF',    # Total basement area (sq ft)
    'Garage Cars',      # Garage size in car capacity
    'Full Bath',        # Number of full bathrooms above ground
    'TotRms AbvGrd',    # Total rooms above ground
    'Fireplaces'        # Number of fireplaces
]
target = 'SalePrice'

df_mlr = df[features + [target]].dropna()

print("Dataset Shape:", df_mlr.shape)
print("\n--- First 5 Rows ---")
display(df_mlr.head())

print("\n--- Statistical Summary ---")
display(df_mlr.describe())

## Preprocessing

For MLR we need more careful preprocessing than SLR:
1. **Check for missing values** — fill with column medians
2. **Check multicollinearity** — correlation heatmap before training
3. **Scale all features** — important when features are on different scales (sq ft vs count)
4. **Train/test split** — 80/20, same random state as Topic 3 for fair comparison

In [ ]:
# Step 1: Fill any remaining missing values with column median
for col in features:
    df_mlr[col].fillna(df_mlr[col].median(), inplace=True)

print("Missing values after fill:", df_mlr.isnull().sum().sum())

# Step 2: Correlation heatmap — check for multicollinearity BEFORE training
# INTERVIEW NOTE: Features with correlation > 0.8 with each other can cause instability
plt.figure(figsize=(10, 7))
corr_matrix = df_mlr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show only lower triangle
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix — Checking Multicollinearity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Step 3: Separate features and target
X = df_mlr[features].values
y = df_mlr[target].values

# Step 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Scale features — fit only on training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTraining set : {X_train_scaled.shape}")
print(f"Test set     : {X_test_scaled.shape}")
print(f"Features     : {len(features)}")
print(f"\nBaseline RMSE (predict mean always): ${np.std(y_test):,.0f}")

## Part 2: From Scratch Implementation

We implement Multiple Linear Regression using the **Normal Equation** in full matrix form:

```
β = (XᵀX)⁻¹ Xᵀy
```

This is the same equation we used in Topic 2's overview, but now we build the full class with proper coefficient interpretation. The key step is prepending a column of 1s to X — this allows β₀ (the intercept) to be learned as part of the same matrix operation.

In [ ]:
class MultipleLinearRegressionScratch:
    """
    Multiple Linear Regression from scratch using the Normal Equation.
    β = (XᵀX)⁻¹ Xᵀy
    Handles p >= 1 features — generalizes SLR to full MLR.
    """

    def __init__(self):
        self.beta = None       # Full coefficient vector [β₀, β₁, ..., βₚ]
        self.beta_0 = None     # Intercept (extracted separately for clarity)
        self.beta_coefs = None # Feature coefficients [β₁, ..., βₚ]

    def fit(self, X, y):
        n_samples = X.shape[0]

        # Step 1: Prepend a column of 1s to X for the intercept term β₀
        # Shape changes from (n, p) to (n, p+1)
        # INTERVIEW NOTE: Without this, model has no intercept — line forced through origin
        X_b = np.c_[np.ones((n_samples, 1)), X]

        # Step 2: Apply the Normal Equation: β = (XᵀX)⁻¹ Xᵀy
        # XᵀX is shape (p+1, p+1) — square, should be invertible if no perfect multicollinearity
        # INTERVIEW NOTE: np.linalg.inv is O(p³) — expensive for large p
        # For large p, use np.linalg.lstsq instead (more numerically stable)
        XtX = X_b.T @ X_b          # (p+1, p+1) matrix
        Xty = X_b.T @ y            # (p+1,) vector
        self.beta = np.linalg.inv(XtX) @ Xty  # (p+1,) solution vector

        # Extract intercept and feature coefficients separately
        self.beta_0    = self.beta[0]
        self.beta_coefs = self.beta[1:]

    def predict(self, X):
        # Add bias column then compute ŷ = Xβ
        n_samples = X.shape[0]
        X_b = np.c_[np.ones((n_samples, 1)), X]
        return X_b @ self.beta

    def score(self, X, y):
        # R² score
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

    def adjusted_r2(self, X, y):
        # Adjusted R² — penalizes extra features
        # Formula: 1 - (1 - R²) × (n - 1) / (n - p - 1)
        n = X.shape[0]
        p = X.shape[1]
        r2 = self.score(X, y)
        return 1 - (1 - r2) * (n - 1) / (n - p - 1)

# Train the model
mlr_scratch = MultipleLinearRegressionScratch()
mlr_scratch.fit(X_train_scaled, y_train)

print("=== MLR From Scratch — Learned Coefficients ===")
print(f"β₀ (Intercept) : {mlr_scratch.beta_0:>12,.2f}")
print()
for feat, coef in zip(features, mlr_scratch.beta_coefs):
    print(f"  {feat:20s}: {coef:>12,.2f}")

In [ ]:
# Evaluate the from-scratch MLR model
y_pred_scratch = mlr_scratch.predict(X_test_scaled)

mse_s   = mean_squared_error(y_test, y_pred_scratch)
rmse_s  = np.sqrt(mse_s)
mae_s   = mean_absolute_error(y_test, y_pred_scratch)
r2_s    = r2_score(y_test, y_pred_scratch)
adj_r2_s = mlr_scratch.adjusted_r2(X_test_scaled, y_test)

print("=== From Scratch MLR — Test Set Evaluation ===")
print(f"MSE         : {mse_s:>20,.2f}")
print(f"RMSE        : ${rmse_s:>18,.2f}")
print(f"MAE         : ${mae_s:>18,.2f}")
print(f"R²          : {r2_s:>20.4f}")
print(f"Adjusted R² : {adj_r2_s:>20.4f}")
print(f"\nRecall Topic 3 SLR used only 1 feature. MLR with {len(features)} features should show higher R².")

## Part 3: Sklearn Implementation

Scikit-learn's `LinearRegression` uses LAPACK's least-squares solver internally — more numerically stable than direct matrix inversion for large feature sets. The results should match our scratch implementation closely.

We also compute **VIF (Variance Inflation Factor)** manually to detect multicollinearity. VIF for feature j measures how much the variance of βⱼ is inflated due to correlation with other features:
- VIF = 1: no correlation
- VIF 1–5: moderate, acceptable
- VIF > 10: severe multicollinearity — problem

In [ ]:
# Train sklearn MLR
mlr_sklearn = LinearRegression()
mlr_sklearn.fit(X_train_scaled, y_train)
y_pred_sklearn = mlr_sklearn.predict(X_test_scaled)

rmse_sk = np.sqrt(mean_squared_error(y_test, y_pred_sklearn))
r2_sk   = r2_score(y_test, y_pred_sklearn)
n_test  = X_test_scaled.shape[0]
p       = X_test_scaled.shape[1]
adj_r2_sk = 1 - (1 - r2_sk) * (n_test - 1) / (n_test - p - 1)

print("=== Sklearn MLR Results ===")
print(f"RMSE        : ${rmse_sk:,.2f}")
print(f"R²          : {r2_sk:.4f}")
print(f"Adjusted R² : {adj_r2_sk:.4f}")

print("\n=== Scratch vs Sklearn Coefficient Comparison ===")
coef_df = pd.DataFrame({
    'Feature'        : features,
    'Scratch β'      : mlr_scratch.beta_coefs.round(2),
    'Sklearn β'      : mlr_sklearn.coef_.round(2),
    'Match'          : ['✅' if abs(a - b) < 1 else '❌'
                        for a, b in zip(mlr_scratch.beta_coefs, mlr_sklearn.coef_)]
})
display(coef_df)

# --- VIF Calculation from scratch (no statsmodels dependency) ---
# VIF for feature j = 1 / (1 - R²_j)
# where R²_j is R² from regressing feature j on all other features
print("\n=== Variance Inflation Factor (VIF) ===")
vif_values = []
for j in range(X_train_scaled.shape[1]):
    # Target: feature j; Predictors: all other features
    X_other = np.delete(X_train_scaled, j, axis=1)
    y_feat  = X_train_scaled[:, j]
    reg = LinearRegression()
    reg.fit(X_other, y_feat)
    r2_j = r2_score(y_feat, reg.predict(X_other))
    # Avoid division by zero if R²=1 (perfect multicollinearity)
    vif = 1 / (1 - r2_j) if r2_j < 1.0 else float('inf')
    vif_values.append(round(vif, 2))

vif_df = pd.DataFrame({'Feature': features, 'VIF': vif_values})
vif_df['Status'] = vif_df['VIF'].apply(
    lambda v: '✅ OK' if v < 5 else ('⚠️ Moderate' if v < 10 else '❌ High')
)
display(vif_df)
print("VIF > 10 = severe multicollinearity. VIF < 5 = acceptable.")

In [ ]:
# --- Visualisation 1: Actual vs Predicted + Feature Coefficients ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred_scratch, alpha=0.4, color='steelblue', s=20)
min_val = min(y_test.min(), y_pred_scratch.min())
max_val = max(y_test.max(), y_pred_scratch.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title(f'Actual vs Predicted\nR² = {r2_s:.4f} | Adj R² = {adj_r2_s:.4f}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Sale Price ($)', fontsize=11)
axes[0].set_ylabel('Predicted Sale Price ($)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Feature coefficients — which features contribute most?
# INTERVIEW NOTE: Coefficients are comparable only after scaling — largest absolute value = strongest effect
coefs = mlr_scratch.beta_coefs
colors = ['steelblue' if c > 0 else 'salmon' for c in coefs]
axes[1].barh(features, coefs, color=colors, edgecolor='black')
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].set_title('Feature Coefficients (after scaling)\nBlue = positive effect, Red = negative effect',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coefficient Value (β)', fontsize=11)
axes[1].set_ylabel('Feature', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Multiple Linear Regression — Ames Housing Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Visualisation 2: Residual Plot ---
residuals = y_test - y_pred_scratch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_scratch, residuals, alpha=0.4, color='purple', s=20)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residuals vs Predicted Values', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Sale Price ($)', fontsize=11)
axes[0].set_ylabel('Residual ($)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residual Distribution\n(Should be approximately Normal)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual Value ($)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('MLR Residual Analysis — Checking Assumptions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

MLR has no regularization hyperparameter, but a critical practical question is:

**How does the number of features affect R² and Adjusted R²?**

We add features one by one (ordered by correlation with target) and observe:
- Plain R² should always increase or stay the same
- Adjusted R² should increase only when a feature is genuinely useful

This experiment directly shows WHY Adjusted R² exists.

In [ ]:
# Order features by absolute correlation with target
correlations = df_mlr[features].corrwith(df_mlr[target]).abs().sort_values(ascending=False)
ordered_features = correlations.index.tolist()

print("Features ordered by correlation with SalePrice:")
for feat, corr in correlations.items():
    print(f"  {feat:20s}: {corr:.4f}")

# Add features one by one and track R² vs Adjusted R²
r2_scores    = []
adj_r2_scores = []
feature_counts = list(range(1, len(ordered_features) + 1))

for k in feature_counts:
    selected = ordered_features[:k]
    X_k = df_mlr[selected].values

    X_tr, X_te, y_tr, y_te = train_test_split(X_k, y, test_size=0.2, random_state=42)

    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)

    model = MultipleLinearRegressionScratch()
    model.fit(X_tr, y_tr)

    r2    = model.score(X_te, y_te)
    adj_r2 = model.adjusted_r2(X_te, y_te)

    r2_scores.append(r2)
    adj_r2_scores.append(adj_r2)

# Plot R² vs Adjusted R² as features increase
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(feature_counts, r2_scores, marker='o', color='blue',
        linewidth=2, markersize=8, label='R²')
ax.plot(feature_counts, adj_r2_scores, marker='s', color='orange',
        linewidth=2, markersize=8, label='Adjusted R²')
ax.set_title('Effect of Number of Features on R² vs Adjusted R²',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Features', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_xticks(feature_counts)
ax.set_xticklabels([f"{k}\n({ordered_features[k-1].split()[0]})" for k in feature_counts],
                    fontsize=8)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey observation:")
print(f"R² with 1 feature  : {r2_scores[0]:.4f}")
print(f"R² with {len(features)} features : {r2_scores[-1]:.4f}  ← always increases")
print(f"Adj R² with 1 feat : {adj_r2_scores[0]:.4f}")
print(f"Adj R² with {len(features)} feat : {adj_r2_scores[-1]:.4f}  ← only increases if feature helps")

## Part 5: Interview Corner

**Question: What is multicollinearity, how do you detect it, and how do you fix it?**

This is one of the most frequently asked MLR questions at FAANG interviews. Here is the complete answer:

**What it is:**  
Multicollinearity occurs when two or more input features are highly correlated with each other. For example, `TotalRooms` and `GrLivArea` both measure house size — they carry overlapping information.

**Why it hurts:**  
The Normal Equation requires inverting (XᵀX). When features are correlated, (XᵀX) becomes near-singular — its determinant approaches zero and its inverse blows up. This causes:
- Coefficient estimates become huge and unstable
- Coefficients flip signs when you add/remove a single data point
- Standard errors explode — hypothesis tests become meaningless
- The model still predicts well but cannot be interpreted

**How to detect it:**
1. Correlation heatmap — look for pairs with |correlation| > 0.8
2. VIF > 10 for any feature signals severe multicollinearity

**How to fix it:**
1. Remove one feature from each highly correlated pair
2. Apply PCA to create uncorrelated components
3. Use **Ridge Regression** — adds αI to (XᵀX), making (XᵀX + αI) always invertible

**Key insight for interviews:**  
Multicollinearity hurts interpretability, not necessarily prediction accuracy. The model can still predict well even with multicollinear features — it just cannot tell you which feature is responsible for the prediction.

## Key Takeaways

Remember these 5 bullets for placement interviews:

- **MLR extends SLR to p features using matrix algebra**: ŷ = Xβ, solved by β = (XᵀX)⁻¹Xᵀy. Each βⱼ is the partial effect of feature j holding all others constant — not the same as the SLR coefficient for that feature alone.
- **Multicollinearity is MLR's biggest weakness**: Correlated features make (XᵀX) near-singular, destabilizing coefficients. Detect with correlation heatmap or VIF > 10. Fix with Ridge regression or feature removal.
- **Always use Adjusted R² to compare models**: Plain R² always increases with more features. Adjusted R² penalizes unnecessary features and is the correct metric when comparing models with different feature counts.
- **VIF measures each feature's multicollinearity**: VIF for feature j = 1/(1 - R²_j) where R²_j comes from regressing feature j on all others. VIF < 5 is fine; VIF > 10 is a serious problem.
- **More features can hurt, not just help**: Adding irrelevant features increases variance, destabilizes coefficients, and can lower Adjusted R². Feature selection (domain knowledge, correlation analysis, Lasso) is as important as the algorithm itself.